
# OPTIMAL WEEKLY ROUTING WITH GOOGLE OR-TOOLS
Constraints:
- 20 employees working over 5 days (Monday-Friday)
- Max 3 clients per employee per day
- Max 6 hours work time per day (travel + care hours)
- Compatibility based on pets (dogs/cats) and smoking
- **Client availability: clients are only scheduled on days they are available**
- Interactive Map with day selection + detailed schedule table per employee per day


# 1. Import Libraries

In [1]:
import json
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
import folium
from scipy.spatial import cKDTree
from ortools.constraint_solver import routing_enums_pb2, pywrapcp
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
from ipywidgets import interact, Dropdown
from IPython.display import IFrame, display, HTML

print('Bibliotheken geladen.')

Bibliotheken geladen.


# 2. Wegennet laden en graph bouwen

In [2]:
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f'Aantal edges: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

G = nx.Graph()
node_coords = {}
edge_geom = {}

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    u, v = row['u'], row['v']
    G.add_edge(u, v, weight=row['travel_time_min'], geometry=geom)
    node_coords[u] = (coords[0][0], coords[0][1])
    node_coords[v] = (coords[-1][0], coords[-1][1])
    edge_geom[(u, v)] = geom
    edge_geom[(v, u)] = geom

print(f'Graph: {G.number_of_nodes()} knopen, {G.number_of_edges()} takken.')

node_ids = list(node_coords.keys())
node_lons_arr = np.array([node_coords[n][0] for n in node_ids])
node_lats_arr = np.array([node_coords[n][1] for n in node_ids])
kd_tree = cKDTree(np.column_stack((node_lons_arr, node_lats_arr)))

def nearest_node(lon, lat):
    _, idx = kd_tree.query([lon, lat])
    return node_ids[idx]

Aantal edges: 7183
Graph: 3120 knopen, 4340 takken.


# 3. Medewerkers laden (inclusief huisdier- en rookvoorkeuren)

In [3]:
employee_data = [
    ('employees 1',  50.8872, 5.9812),
    ('employees 2',  50.8895, 5.9820),
    ('employees 3',  50.8883, 5.9830),
    ('employees 4',  50.8855, 5.9795),
    ('employees 5',  50.8945, 5.9660),
    ('employees 6',  50.8878, 5.9808),
    ('employees 7',  50.8948, 5.9700),
    ('employees 8',  50.8870, 5.9825),
    ('employees 9',  50.8868, 5.9817),
    ('employees 10', 50.8785, 5.9750),
    ('employees 11', 50.8840, 5.9810),
    ('employees 12', 50.8860, 5.9835),
    ('employees 13', 50.8850, 5.9880),
    ('employees 14', 50.8890, 5.9822),
    ('employees 15', 50.8710, 5.9920),
    ('employees 16', 50.8810, 5.9680),
    ('employees 17', 50.8952, 5.9672),
    ('employees 18', 50.8875, 5.9805),
    ('employees 19', 50.8940, 5.9665),
    ('employees 20', 50.8790, 5.9760),
]
employees_df = pd.DataFrame(employee_data, columns=['name', 'lat', 'lon'])
employees_df['node'] = employees_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)

emp_extra = pd.read_csv('../output/employees.csv')
emp_extra['name'] = emp_extra['name'].str.strip()
employees_df = employees_df.merge(emp_extra[['name', 'dogs', 'cats', 'smokes']], on='name', how='left')
employees_df['dogs'] = employees_df['dogs'].fillna(-1).astype(int)
employees_df['cats'] = employees_df['cats'].fillna(-1).astype(int)
employees_df['smokes'] = employees_df['smokes'].fillna(False).astype(bool)

print(f'Aantal medewerkers: {len(employees_df)}')

Aantal medewerkers: 20


# 4. Cliënten laden (coördinaten, zorgtijd, huisdieren, rook, tijdvenster)

In [4]:
clients_df = pd.read_csv('../output/clients_availability.csv')

# Coördinaten detecteren
coord_col = None
for col in clients_df.columns:
    sample = clients_df[col].dropna().astype(str).iloc[0]
    parts = sample.replace(',', ' ').replace(';', ' ').split()
    if len(parts) == 2:
        try:
            float(parts[0]); float(parts[1])
            coord_col = col
            break
        except:
            pass
coord_col = coord_col or clients_df.columns[0]

def split_coords(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    return (float(parts[0]), float(parts[1])) if len(parts) == 2 else (np.nan, np.nan)

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(lambda x: pd.Series(split_coords(x)))
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
clients_df['client_id'] = clients_df.index
clients_df['node'] = clients_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)

# Kolommen standaard invullen
for col in ['dogs', 'cats', 'smokes', 'care_hours']:
    if col not in clients_df.columns:
        clients_df[col] = 0 if col in ['dogs', 'cats'] else (False if col == 'smokes' else 1.0)
clients_df['smokes'] = clients_df['smokes'].astype(bool)
clients_df['care_hours'] = pd.to_numeric(clients_df['care_hours'], errors='coerce').fillna(1.0)

# Tijdvenster omzetten naar minuten na 07:00
def window_to_minutes(t_str):
    h, m = map(int, t_str.split(':'))
    return h * 60 + m - 420  # 07:00 = 0
clients_df['tw_min'] = clients_df['time_window_start'].apply(window_to_minutes)
clients_df['tw_max'] = clients_df['time_window_end'].apply(window_to_minutes)

# ── Beschikbaarheid laden: unavailable_days -> beschikbaar per dag (0=ma..4=vr) ──
DAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

def parse_unavailable_days(val):
    """Geeft een set van dag-indices (0=ma, 4=vr) waarop cliënt NIET beschikbaar is."""
    if pd.isna(val) or str(val).strip() == '':
        return set()
    days = set()
    for part in str(val).split(','):
        part = part.strip()
        if part in DAY_NAMES:
            days.add(DAY_NAMES.index(part))
    return days

clients_df['unavailable_day_indices'] = clients_df['unavailable_days'].apply(parse_unavailable_days)

print(f'Aantal cliënten: {len(clients_df)}')
print(f'Voorbeeld beschikbaarheid:')
for _, row in clients_df.head(5).iterrows():
    unavail = [DAY_NAMES[d] for d in sorted(row['unavailable_day_indices'])]
    avail   = [DAY_NAMES[d] for d in range(5) if d not in row['unavailable_day_indices']]
    print(f"  {row['name']}: niet beschikbaar op {unavail}, beschikbaar op {avail}")


Aantal cliënten: 100
Voorbeeld beschikbaarheid:
  Client 1: niet beschikbaar op ['Tuesday'], beschikbaar op ['Monday', 'Wednesday', 'Thursday', 'Friday']
  Client 2: niet beschikbaar op ['Wednesday', 'Friday'], beschikbaar op ['Monday', 'Tuesday', 'Thursday']
  Client 3: niet beschikbaar op ['Thursday'], beschikbaar op ['Monday', 'Tuesday', 'Wednesday', 'Friday']
  Client 4: niet beschikbaar op ['Monday', 'Wednesday'], beschikbaar op ['Tuesday', 'Thursday', 'Friday']
  Client 5: niet beschikbaar op ['Friday'], beschikbaar op ['Monday', 'Tuesday', 'Wednesday', 'Thursday']


# 5. Reistijdenmatrix berekenen

In [5]:
N_EMPLOYEES = len(employees_df)
N_CLIENTS = len(clients_df)
N_TOTAL = N_EMPLOYEES + N_CLIENTS   # 20 + 100 = 120
SCALE = 100

all_nodes = employees_df['node'].tolist() + clients_df['node'].tolist()
unique_sources = list(set(all_nodes))
print(f'Dijkstra uitvoeren vanaf {len(unique_sources)} unieke knopen ...')

dist_from = {}
for i, src in enumerate(unique_sources):
    dist_from[src] = nx.single_source_dijkstra_path_length(G, src, weight='weight')
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(unique_sources)} gereed')

time_matrix = np.zeros((N_TOTAL, N_TOTAL), dtype=np.int64)
for i in range(N_TOTAL):
    src_graph = all_nodes[i]
    lengths = dist_from[src_graph]
    for j in range(N_TOTAL):
        dst_graph = all_nodes[j]
        t = lengths.get(dst_graph, float('inf'))
        time_matrix[i][j] = int(t * SCALE) if t != float('inf') else 10_000_000

print(f'Reistijdmatrix: {time_matrix.shape}')
print(f'Min reistijd: {time_matrix[time_matrix>0].min()/SCALE:.1f} min')
print(f'Max reistijd: {time_matrix[time_matrix<10_000_000].max()/SCALE:.1f} min')

Dijkstra uitvoeren vanaf 102 unieke knopen ...
  20/102 gereed
  40/102 gereed
  60/102 gereed
  80/102 gereed
  100/102 gereed
Reistijdmatrix: (120, 120)
Min reistijd: 0.0 min
Max reistijd: 11.2 min


# 6. Meerdaags model instellen (100 voertuigen: 20 medewerkers × 5 dagen)

In [6]:
NUM_DAYS = 5
MAX_CLIENTS_PER_VEHICLE = 3
MAX_WORK_MINUTES = 360  # 6 uur

vehicles = []
for emp_id in range(N_EMPLOYEES):
    for day in range(NUM_DAYS):
        vehicles.append({
            'vehicle_id': len(vehicles),
            'emp_id': emp_id,
            'day': day,
            'start_node': emp_id,
            'end_node': emp_id
        })
N_VEHICLES = len(vehicles)
print(f'Aantal voertuigen (medewerker×dag): {N_VEHICLES}')

Aantal voertuigen (medewerker×dag): 100


# 7. OR-Tools data model (tijdvensters, capaciteit, werktijdlimiet)

In [7]:
data = {}
data['time_matrix'] = time_matrix.tolist()
data['num_vehicles'] = N_VEHICLES
data['starts'] = [v['start_node'] for v in vehicles]
data['ends']   = [v['end_node'] for v in vehicles]
data['demands'] = [0] * N_EMPLOYEES + [1] * N_CLIENTS
data['capacities'] = [MAX_CLIENTS_PER_VEHICLE] * N_VEHICLES

# Zorgtijd in minuten (alleen voor cliënten, voor thuis = 0)
service_time = [0] * N_EMPLOYEES + (clients_df['care_hours'] * 60).round().astype(int).tolist()

# Tijdvensters: (start, end) in minuten na 07:00
time_windows = [(0, 660)] * N_EMPLOYEES  # thuis: 07:00 - 18:00
for _, client in clients_df.iterrows():
    time_windows.append((client['tw_min'], client['tw_max']))

manager = pywrapcp.RoutingIndexManager(
    len(data['time_matrix']),
    data['num_vehicles'],
    data['starts'],
    data['ends']
)
routing = pywrapcp.RoutingModel(manager)

# --- Callback voor reistijd + zorgtijd (zorgtijd wordt opgeteld bij vertrek van een knoop) ---
def total_time_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    travel = data['time_matrix'][from_node][to_node]           # al geschaald met SCALE
    service = service_time[from_node] * SCALE                  # zorgtijd ook schalen
    return travel + service

transit_callback = routing.RegisterTransitCallback(total_time_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback)

# --- Capaciteitsdimensie (max 3 cliënten per voertuig) ---
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 0, data['capacities'], True, 'Capacity'
)

# --- Tijdsdimensie (bevat reistijd + zorgtijd) ---
routing.AddDimension(
    transit_callback,          # transit callback met reistijd+zorgtijd
    0,                         # slack
    660 * SCALE,               # maximum totaal (18:00) – later per voertuig beperkt tot 6 uur
    False,                     # geen start slack
    'Time'
)
time_dim = routing.GetDimensionOrDie('Time')

# Tijdvensters instellen (arrival time moet binnen venster vallen)
for node in range(N_TOTAL):
    index = manager.NodeToIndex(node)
    tw_min, tw_max = time_windows[node]
    time_dim.CumulVar(index).SetRange(tw_min * SCALE, tw_max * SCALE)

# Maximale werktijd per voertuig instellen (360 minuten = 6 uur)
MAX_WORK_SCALED = MAX_WORK_MINUTES * SCALE
for v in range(N_VEHICLES):
    start_index = routing.Start(v)
    end_index = routing.End(v)
    # De totale tijd van start tot eind (inclusief alle reistijd en zorgtijd) mag niet > 360 minuten zijn
    time_dim.SetSpanUpperBoundForVehicle(MAX_WORK_SCALED, v)

print('Data model klaar (zorgtijd zit in de tijdsdimensie, max 6 uur per dag).')

Data model klaar (zorgtijd zit in de tijdsdimensie, max 6 uur per dag).


# 8. Compatibiliteit (huisdieren/rook) en voertuigbeperkingen

In [8]:
compatible_emp_per_client = []
skipped_clients = []
for cid in range(N_CLIENTS):
    client = clients_df.iloc[cid]
    compatible = []
    for emp_id, emp in employees_df.iterrows():
        if emp['dogs'] != -1 and client['dogs'] > emp['dogs']:
            continue
        if emp['cats'] != -1 and client['cats'] > emp['cats']:
            continue
        if not emp['smokes'] and client['smokes']:
            continue
        compatible.append(emp_id)
    if not compatible:
        skipped_clients.append(cid)
    compatible_emp_per_client.append(compatible)

solver = routing.solver()
for cid in range(N_CLIENTS):
    node_idx = manager.NodeToIndex(N_EMPLOYEES + cid)
    unavail_days = clients_df.iloc[cid]['unavailable_day_indices']

    if cid in skipped_clients:
        routing.AddDisjunction([node_idx], 10_000_000)
    else:
        vehicle_var = routing.VehicleVar(node_idx)
        for v in range(N_VEHICLES):
            emp_id  = vehicles[v]['emp_id']
            day_idx = vehicles[v]['day']
            # Blokkeer als medewerker niet compatibel is
            if emp_id not in compatible_emp_per_client[cid]:
                solver.Add(vehicle_var != v)
            # Blokkeer als cliënt op deze dag niet beschikbaar is
            elif day_idx in unavail_days:
                solver.Add(vehicle_var != v)

print(f'Cliënten zonder geschikte medewerker: {len(skipped_clients)}')
# Rapporteer cliënten met beperkte beschikbaarheid
limited = [(cid, clients_df.iloc[cid]['name'], sorted(clients_df.iloc[cid]['unavailable_day_indices']))
           for cid in range(N_CLIENTS) if clients_df.iloc[cid]['unavailable_day_indices']]
print(f'Cliënten met dagbeperkingen: {len(limited)}')
print('Eerste 5 voorbeelden:')
for cid, name, days in limited[:5]:
    print(f'  {name}: niet beschikbaar op {[DAY_NAMES[d] for d in days]}')


Cliënten zonder geschikte medewerker: 33
Cliënten met dagbeperkingen: 100
Eerste 5 voorbeelden:
  Client 1: niet beschikbaar op ['Tuesday']
  Client 2: niet beschikbaar op ['Wednesday', 'Friday']
  Client 3: niet beschikbaar op ['Thursday']
  Client 4: niet beschikbaar op ['Monday', 'Wednesday']
  Client 5: niet beschikbaar op ['Friday']


# 9. VRP oplossen

In [9]:
search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
)
search_params.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_params.time_limit.seconds = 180
search_params.log_search = True

print('Meerdaagse VRP wordt opgelost (max 3 cliënten/dag, max 6u/dag, tijdvensters)...')
solution = routing.SolveWithParameters(search_params)

if solution:
    print(f'\nOplossing gevonden! Totale reistijd: {solution.ObjectiveValue() / SCALE:.1f} min')
else:
    print('\nGeen oplossing gevonden.')

Meerdaagse VRP wordt opgelost (max 3 cliënten/dag, max 6u/dag, tijdvensters)...

Oplossing gevonden! Totale reistijd: 12111.7 min


# 10. Routes extraheren

In [10]:
def extract_routes_safe(solution, routing, manager):
    routes = []
    for v in range(N_VEHICLES):
        try:
            index = routing.Start(v)
            nodes = []
            # Verzamel nodes tot het einde
            while not routing.IsEnd(index):
                node = manager.IndexToNode(index)
                nodes.append(node)
                index = solution.Value(routing.NextVar(index))
            nodes.append(manager.IndexToNode(index))
            
            # Alleen cliënten (node >= N_EMPLOYEES)
            client_ids = [n - N_EMPLOYEES for n in nodes if n >= N_EMPLOYEES]
            if not client_ids:
                # Geen cliënten: voeg toch een lege route toe? Nee, overslaan
                continue
            
            # Haal totale werktijd uit de Time dimensie
            time_dim = routing.GetDimensionOrDie('Time')
            start_cumul = solution.Value(time_dim.CumulVar(routing.Start(v)))
            end_cumul = solution.Value(time_dim.CumulVar(routing.End(v)))
            work_time = (end_cumul - start_cumul) / SCALE
            
            # Reistijd (exclusief zorgtijd) apart berekenen (optioneel)
            travel_time = 0.0
            for i in range(len(nodes)-1):
                travel_time += time_matrix[nodes[i]][nodes[i+1]] / SCALE
            
            routes.append({
                'vehicle_id': v,
                'emp_id': vehicles[v]['emp_id'],
                'day': vehicles[v]['day'],
                'nodes': nodes,
                'client_ids': client_ids,
                'work_time': work_time,
                'travel_time': travel_time
            })
        except Exception as e:
            print(f"Waarschuwing: route voor voertuig {v} kon niet worden uitgelezen: {e}")
            continue
    return routes

if solution:
    all_routes = extract_routes_safe(solution, routing, manager)
    print(f'{len(all_routes)} routes geëxtraheerd (alleen routes met minimaal 1 cliënt).')
else:
    all_routes = []
    print('Geen oplossing beschikbaar.')

41 routes geëxtraheerd (alleen routes met minimaal 1 cliënt).


# 11. Overzicht van niet-ingeplande cliënten

In [11]:
if solution:
    geplande_clients = set()
    for r in all_routes:
        geplande_clients.update(r['client_ids'])
    alle_clients = set(range(N_CLIENTS))
    niet_gepland = alle_clients - geplande_clients
    if niet_gepland:
        print(f'\nNiet ingeplande cliënten ({len(niet_gepland)}):')
        for cid in sorted(niet_gepland):
            c = clients_df.iloc[cid]
            reden = 'geen geschikte medewerker' if cid in skipped_clients else 'capaciteit/tijd te krap'
            print(f'  Cliënt {cid} ({c["name"] if "name" in c else ""}): {reden}')
    else:
        print('\nAlle cliënten zijn ingepland!')


Alle cliënten zijn ingepland!


# 12. Samenvatting per medewerker per dag

In [12]:
if solution:
    print('\n=== Samenvatting per medewerker per dag ===')
    for day in range(NUM_DAYS):
        dag_str = ['maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag'][day]
        print(f'\n--- Dag {day+1} ({dag_str}) ---')
        dag_routes = [r for r in all_routes if r['day'] == day and r['client_ids']]
        for r in dag_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            print(f'{emp_name:15s}: {len(r["client_ids"])} cliënten | werktijd {r["work_time"]:.1f} min | reistijd {r["travel_time"]:.1f} min | stops: {r["client_ids"]}')


=== Samenvatting per medewerker per dag ===

--- Dag 1 (maandag) ---
employees 3    : 2 cliënten | werktijd 243.4 min | reistijd 3.4 min | stops: [91, 10]
employees 4    : 3 cliënten | werktijd 332.1 min | reistijd 2.1 min | stops: [61, 87, 73]
employees 5    : 3 cliënten | werktijd 340.9 min | reistijd 10.9 min | stops: [1, 32, 47]
employees 6    : 2 cliënten | werktijd 300.0 min | reistijd 0.0 min | stops: [16, 77]
employees 7    : 3 cliënten | werktijd 247.6 min | reistijd 7.6 min | stops: [25, 14, 28]
employees 12   : 2 cliënten | werktijd 331.0 min | reistijd 1.0 min | stops: [0, 90]
employees 13   : 3 cliënten | werktijd 337.4 min | reistijd 7.4 min | stops: [5, 92, 3]
employees 14   : 3 cliënten | werktijd 335.2 min | reistijd 5.2 min | stops: [19, 2, 13]
employees 15   : 3 cliënten | werktijd 276.5 min | reistijd 6.5 min | stops: [42, 34, 48]
employees 17   : 3 cliënten | werktijd 253.6 min | reistijd 13.6 min | stops: [68, 80, 56]
employees 19   : 1 cliënten | werktijd 120.4 

# 13. Interactieve kaart (dropdown voor dagen)

In [13]:
if solution:
    import json as _json
    from IPython.display import display, HTML
    from datetime import datetime, timedelta

    EMPLOYEE_COLORS = [
        '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
        '#469990','#dcbeff','#9A6324','#ff8c00','#800000',
        '#aaffc3','#808000','#00bfff','#000075','#808080',
    ]
    DAY_NAMES_NL = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag']
    DAY_NAMES_EN = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']
    START_OF_DAY = datetime.strptime('08:00', '%H:%M')

    def minutes_to_time(base, minutes):
        return (base + timedelta(minutes=minutes)).strftime('%H:%M')

    def build_schedule(route):
        vid        = route['emp_id']
        client_ids = route['client_ids']
        rows       = []
        current_time = 0.0
        rows.append({'Stop': 'Thuis (vertrek)', 'Cliënt': '—', 'Aankomst': '—',
                     'Zorgtijd': '—', 'Vertrek': minutes_to_time(START_OF_DAY, current_time)})
        prev_idx = vid
        for step, cid in enumerate(client_ids):
            client_matrix_idx = N_EMPLOYEES + cid
            travel = time_matrix[prev_idx][client_matrix_idx] / SCALE
            current_time += travel
            arrival = minutes_to_time(START_OF_DAY, current_time)
            care_hours = float(clients_df.loc[cid, 'care_hours']) if 'care_hours' in clients_df.columns else 1.0
            care_min   = care_hours * 60.0
            current_time += care_min
            departure  = minutes_to_time(START_OF_DAY, current_time)
            client_name = clients_df.loc[cid, 'name'] if 'name' in clients_df.columns else f'Cliënt {cid}'
            rows.append({'Stop': f'Stop {step+1}', 'Cliënt': client_name,
                         'Aankomst': arrival,
                         'Zorgtijd': f'{int(care_min)} min ({care_hours:.1f} uur)',
                         'Vertrek': departure})
            prev_idx = client_matrix_idx
        travel_home  = time_matrix[prev_idx][vid] / SCALE
        current_time += travel_home
        rows.append({'Stop': 'Thuis (terug)', 'Cliënt': '—',
                     'Aankomst': minutes_to_time(START_OF_DAY, current_time),
                     'Zorgtijd': '—', 'Vertrek': '—'})
        return rows

    def schedule_to_html(schedule_rows, emp_color):
        html = (
            '<table style="width:100%;border-collapse:collapse;font-size:12px;margin-top:0;">'
            '<thead><tr style="background:' + emp_color + ';color:#fff;">'
            '<th style="padding:5px 8px;text-align:left;">Stop</th>'
            '<th style="padding:5px 8px;text-align:left;">Cliënt</th>'
            '<th style="padding:5px 8px;text-align:left;">Aankomst</th>'
            '<th style="padding:5px 8px;text-align:left;">Zorgtijd</th>'
            '<th style="padding:5px 8px;text-align:left;">Vertrek</th>'
            '</tr></thead><tbody>'
        )
        for i, row in enumerate(schedule_rows):
            bg = '#f4f4f4' if i % 2 == 0 else '#ffffff'
            html += (
                f'<tr style="background:{bg};">'
                f'<td style="padding:4px 8px;color:#555;">{row["Stop"]}</td>'
                f'<td style="padding:4px 8px;font-weight:bold;">{row["Cliënt"]}</td>'
                f'<td style="padding:4px 8px;">{row["Aankomst"]}</td>'
                f'<td style="padding:4px 8px;">{row["Zorgtijd"]}</td>'
                f'<td style="padding:4px 8px;">{row["Vertrek"]}</td>'
                '</tr>'
            )
        html += '</tbody></table>'
        return html

    def nodes_to_latlon(path_nodes):
        latlon = []
        for i in range(len(path_nodes)-1):
            u, v = path_nodes[i], path_nodes[i+1]
            geom = edge_geom.get((u, v))
            if geom is None:
                cu = node_coords.get(u); cv = node_coords.get(v)
                if cu: latlon.append((cu[1], cu[0]))
                if cv: latlon.append((cv[1], cv[0]))
                continue
            coords = list(geom.coords)
            cu = node_coords.get(u)
            if cu and len(coords) >= 2:
                if abs(coords[-1][0]-cu[0]) < abs(coords[0][0]-cu[0]):
                    coords = coords[::-1]
            latlon.extend([(lat, lon) for lon, lat in coords])
        return latlon

    def road_segment(node_a, node_b):
        try:
            path = nx.shortest_path(G, source=node_a, target=node_b, weight='weight')
            return nodes_to_latlon(path)
        except nx.NetworkXNoPath:
            ca, cb = node_coords.get(node_a), node_coords.get(node_b)
            res = []
            if ca: res.append((ca[1], ca[0]))
            if cb: res.append((cb[1], cb[0]))
            return res

    def generate_day_map(day_index):
        day_name  = DAY_NAMES_NL[day_index]
        day_routes = [r for r in all_routes if r['day'] == day_index and r['client_ids']]
        center_lat = float(np.mean(node_lats_arr))
        center_lon = float(np.mean(node_lons_arr))
        m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')
        for _, row in edges_df.iterrows():
            latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
            folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.3).add_to(m)
        for r in day_routes:
            color    = EMPLOYEE_COLORS[r['emp_id'] % len(EMPLOYEE_COLORS)]
            emp_name = employees_df.loc[r['emp_id'], 'name']
            nodes_seq = r['nodes']
            graph_seq = [all_nodes[n] for n in nodes_seq]
            for seg_i in range(len(graph_seq)-1):
                latlon = road_segment(graph_seq[seg_i], graph_seq[seg_i+1])
                if len(latlon) >= 2:
                    folium.PolyLine(locations=latlon, color=color, weight=4, opacity=0.85,
                                    tooltip=f'{emp_name} | {day_name}').add_to(m)
            for stop_idx, cid in enumerate(r['client_ids']):
                client = clients_df.iloc[cid]
                folium.CircleMarker(
                    location=[client['lat'], client['lon']],
                    radius=6, color='white', weight=1.5, fill=True,
                    fill_color=color, fill_opacity=0.9,
                    popup=folium.Popup(
                        f'<b>{client["name"] if "name" in client else f"Cliënt {cid}"}</b><br>'
                        f'Medewerker: {emp_name}<br>Stop {stop_idx+1}', max_width=200),
                    tooltip=f'{emp_name} stop {stop_idx+1}'
                ).add_to(m)
        for emp_id, emp in employees_df.iterrows():
            color    = EMPLOYEE_COLORS[emp_id % len(EMPLOYEE_COLORS)]
            has_route = any(r['emp_id'] == emp_id for r in day_routes)
            folium.Marker(
                location=[emp['lat'], emp['lon']],
                icon=folium.DivIcon(
                    html=f'<div style="width:22px;height:22px;background:{color};border:3px solid white;'
                         f'border-radius:50%;box-shadow:0 2px 6px rgba(0,0,0,.5);"></div>',
                    icon_size=(22,22), icon_anchor=(11,11)),
                popup=folium.Popup(f"{emp['name']}<br>{'Actief' if has_route else 'Geen bezoeken'}", max_width=240),
                tooltip=f"{emp['name']} (thuis)"
            ).add_to(m)
        legend_rows = ''.join(
            f'<tr><td style="padding:2px 4px;"><div style="width:10px;height:10px;background:'
            f'{EMPLOYEE_COLORS[r["emp_id"] % len(EMPLOYEE_COLORS)]};border-radius:2px;"></div></td>'
            f'<td style="padding:2px 6px;"><b>{employees_df.loc[r["emp_id"], "name"]}</b></td>'
            f'<td style="padding:2px 4px;color:#555;">{len(r["client_ids"])} cliënten</td></tr>'
            for r in day_routes
        )
        m.get_root().html.add_child(folium.Element(
            f'<div style="position:fixed;bottom:20px;left:20px;z-index:1000;background:rgba(255,255,255,0.96);'
            f'padding:10px 14px;border-radius:8px;font-size:11px;font-family:sans-serif;'
            f'box-shadow:0 2px 10px rgba(0,0,0,.3);max-height:400px;overflow-y:auto;">'
            f'<b>{day_name}</b><br><span style="color:#777;">{len(day_routes)} actieve medewerkers</span>'
            f'<table style="margin-top:6px;border-collapse:collapse;">{legend_rows}</table></div>'
        ))
        return m

    def generate_schedule_html(day_index):
        day_name   = DAY_NAMES_NL[day_index]
        day_routes = [r for r in all_routes if r['day'] == day_index and r['client_ids']]
        if not day_routes:
            return f'<p style="color:#888;font-family:sans-serif;">Geen routes op {day_name}.</p>'
        html = (
            f'<h2 style="font-family:sans-serif;margin:0 0 4px 0;font-size:16px;">{day_name}</h2>'
            f'<p style="font-family:sans-serif;color:#555;font-size:12px;margin:0 0 12px 0;">'
            f'{len(day_routes)} actieve medewerkers</p>'
        )
        for r in day_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            color    = EMPLOYEE_COLORS[r['emp_id'] % len(EMPLOYEE_COLORS)]
            sched    = build_schedule(r)
            total_care   = sum(float(clients_df.loc[cid, 'care_hours'])*60 for cid in r['client_ids'])
            total_travel = r['travel_time']
            tabel = schedule_to_html(sched, color)
            html += (
                f'<div style="margin-bottom:16px;border-radius:6px;overflow:hidden;'
                f'box-shadow:0 1px 4px rgba(0,0,0,.12);">'
                f'<div style="background:{color};color:#fff;padding:7px 12px;'
                f'font-family:sans-serif;font-size:12px;font-weight:bold;">'
                f'{emp_name} &nbsp;·&nbsp; {len(r["client_ids"])} cliënten '
                f'&nbsp;·&nbsp; reistijd {total_travel:.0f} min '
                f'&nbsp;·&nbsp; zorgtijd {total_care:.0f} min</div>'
                f'{tabel}</div>'
            )
        return html

    # ── Sla kaarten op als aparte HTML-bestanden ──────────────────────────
    for day in range(5):
        m = generate_day_map(day)
        m.save(f'../output/route_{DAY_NAMES_EN[day]}.html')
        print(f'Kaart opgeslagen: route_{DAY_NAMES_EN[day]}.html')

    # ── Sla planningtabellen op als aparte HTML-bestanden ─────────────────
    for day in range(5):
        sched_html = generate_schedule_html(day)
        day_en = DAY_NAMES_EN[day]
        full_html = f'''<!DOCTYPE html>
<html><head><meta charset="UTF-8">
<style>
  body {{ margin: 12px; font-family: sans-serif; background: #fff; }}
  table {{ border-collapse: collapse; }}
</style>
</head><body>{sched_html}</body></html>'''
        with open(f'../output/schedule_{day_en}.html', 'w', encoding='utf-8') as f:
            f.write(full_html)
    print('Planningtabellen opgeslagen.')

    # ── Overzichtspagina met iframe-tabs voor kaart én tabel ──────────────
    tab_buttons = '\n'.join(
        f'        <button class="tab-btn{" active" if i==0 else ""}" onclick="showDay({i})">{DAY_NAMES_NL[i]}</button>'
        for i in range(5)
    )
    overview_html = f'''<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>VRP Routes per dag</title>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ font-family: Arial, sans-serif; background: #f0f0f0; padding: 14px; }}
    h1 {{ font-size: 18px; margin-bottom: 12px; color: #222; }}
    .tabs {{ display: flex; gap: 6px; margin-bottom: 12px; flex-wrap: wrap; }}
    .tab-btn {{
      padding: 8px 18px; background: #ddd; border: none; cursor: pointer;
      font-size: 13px; border-radius: 4px; transition: background .15s;
    }}
    .tab-btn:hover {{ background: #bbb; }}
    .tab-btn.active {{ background: #2563eb; color: #fff; }}
    .panels {{
      display: grid;
      grid-template-columns: 1fr 420px;
      gap: 12px;
      height: calc(100vh - 90px);
    }}
    @media (max-width: 900px) {{
      .panels {{ grid-template-columns: 1fr; height: auto; }}
      .panel {{ height: 60vh; }}
    }}
    .panel {{
      background: #fff;
      border-radius: 6px;
      overflow: hidden;
      box-shadow: 0 1px 6px rgba(0,0,0,.15);
      height: 100%;
    }}
    iframe {{ width: 100%; height: 100%; border: none; display: block; }}
    .schedule-panel {{ overflow-y: auto; padding: 14px; }}
  </style>
</head>
<body>
  <h1>VRP Weekplanning — cliëntbeschikbaarheid inbegrepen</h1>
  <div class="tabs">
{tab_buttons}
  </div>
  <div class="panels">
    <div class="panel">
      <iframe id="mapFrame" src="route_monday.html"></iframe>
    </div>
    <div class="panel schedule-panel" id="scheduleFrame">
      <iframe id="schedIframe" src="schedule_monday.html" style="width:100%;height:100%;border:none;"></iframe>
    </div>
  </div>
  <script>
    const days = {_json.dumps(DAY_NAMES_EN)};
    function showDay(i) {{
      document.getElementById('mapFrame').src   = 'route_'     + days[i] + '.html';
      document.getElementById('schedIframe').src = 'schedule_' + days[i] + '.html';
      document.querySelectorAll('.tab-btn').forEach((b, idx) =>
        b.classList.toggle('active', idx === i));
    }}
  </script>
</body>
</html>'''

    with open('../output/routes_overview.html', 'w', encoding='utf-8') as f:
        f.write(overview_html)
    print('Overzichtspagina opgeslagen: ../output/routes_overview.html')
    display(HTML('<a href="../output/routes_overview.html" target="_blank">Open routes_overview.html</a>'))
else:
    print('Geen routes – geen kaart beschikbaar.')


Kaart opgeslagen: route_monday.html
Kaart opgeslagen: route_tuesday.html
Kaart opgeslagen: route_wednesday.html
Kaart opgeslagen: route_thursday.html
Kaart opgeslagen: route_friday.html
Planningtabellen opgeslagen.
Overzichtspagina opgeslagen: ../output/routes_overview.html


# 14. Dagplanning per medewerker (tekstueel)

In [14]:
# De dagplanning is nu ook zichtbaar in de interactieve HTML (cell hierboven).
# Onderstaande code genereert aanvullend een tekstueel overzicht per dag in de notebook.

if solution:
    from datetime import datetime, timedelta
    START_OF_DAY = datetime.strptime('08:00', '%H:%M')

    def minutes_to_time(base, minutes):
        return (base + timedelta(minutes=minutes)).strftime('%H:%M')

    print('=== Gedetailleerde dagplanning per medewerker per dag ===')
    for day in range(NUM_DAYS):
        dag_str = ['Maandag','Dinsdag','Woensdag','Donderdag','Vrijdag'][day]
        dag_routes = [r for r in all_routes if r['day'] == day and r['client_ids']]
        if not dag_routes:
            continue
        print(f'\n{'═'*72}')
        print(f' {dag_str.upper()} ({len(dag_routes)} medewerkers)')
        print(f'{'═'*72}')
        for r in dag_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            total_care   = sum(float(clients_df.loc[cid, 'care_hours'])*60 for cid in r['client_ids'])
            total_travel = r['travel_time']
            print(f'\n  {emp_name}  |  reistijd: {total_travel:.0f} min  |  zorgtijd: {total_care:.0f} min')
            print(f'  {'-'*60}')
            header = f'  {"Stop":<18}{"Cliënt":<20}{"Aankomst":<10}{"Zorgtijd":<20}{"Vertrek"}'
            print(header)
            # Bouw schedule
            current_time = 0.0
            print(f'  {"Thuis (vertrek)":<18}{"—":<20}{"—":<10}{"—":<20}{minutes_to_time(START_OF_DAY, current_time)}')
            prev_idx = r['emp_id']
            for step, cid in enumerate(r['client_ids']):
                client_matrix_idx = N_EMPLOYEES + cid
                travel = time_matrix[prev_idx][client_matrix_idx] / SCALE
                current_time += travel
                arrival = minutes_to_time(START_OF_DAY, current_time)
                care_hours  = float(clients_df.loc[cid, 'care_hours']) if 'care_hours' in clients_df.columns else 1.0
                care_min    = care_hours * 60.0
                current_time += care_min
                departure   = minutes_to_time(START_OF_DAY, current_time)
                client_name = clients_df.loc[cid, 'name'] if 'name' in clients_df.columns else f'Cliënt {cid}'
                zorgtijd_str = f'{int(care_min)} min ({care_hours:.1f} uur)'
                print(f'  {f"Stop {step+1}":<18}{client_name:<20}{arrival:<10}{zorgtijd_str:<20}{departure}')
                prev_idx = client_matrix_idx
            travel_home   = time_matrix[prev_idx][r['emp_id']] / SCALE
            current_time += travel_home
            print(f'  {"Thuis (terug)":<18}{"—":<20}{minutes_to_time(START_OF_DAY, current_time):<10}')


=== Gedetailleerde dagplanning per medewerker per dag ===

════════════════════════════════════════════════════════════════════════
 MAANDAG (12 medewerkers)
════════════════════════════════════════════════════════════════════════

  employees 3  |  reistijd: 3 min  |  zorgtijd: 240 min
  ------------------------------------------------------------
  Stop              Cliënt              Aankomst  Zorgtijd            Vertrek
  Thuis (vertrek)   —                   —         —                   08:00
  Stop 1            Client 92           08:01     150 min (2.5 uur)   10:31
  Stop 2            Client 11           10:32     90 min (1.5 uur)    12:02
  Thuis (terug)     —                   12:03     

  employees 4  |  reistijd: 2 min  |  zorgtijd: 330 min
  ------------------------------------------------------------
  Stop              Cliënt              Aankomst  Zorgtijd            Vertrek
  Thuis (vertrek)   —                   —         —                   08:00
  Stop 1          

# 15. OR-Tools Zoekproces — Interactieve Animatie
Genereert `vrp_process_animation.html` — een live kaartvisualisatie van het OR-Tools zoekproces.
Elke poging (medewerker × cliënt × dag) is zichtbaar als pijl op de kaart, inclusief de reden van afwijzing.


In [18]:
# ╔══════════════════════════════════════════════════════════╗
# ║  15. OR-Tools Zoekproces — Interactieve Animatie (zonder emoji’s) ║
# ╚══════════════════════════════════════════════════════════╝

import json as _json
import numpy as _np
from IPython.display import display, HTML

# ── Kleurenpalet ──────────────────────────────────────────
EMPLOYEE_COLORS = [
    '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
    '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
    '#469990','#dcbeff','#9A6324','#ff8c00','#800000',
    '#aaffc3','#808000','#00bfff','#000075','#808080'
]
DAY_NAMES_NL = ['Maandag','Dinsdag','Woensdag','Donderdag','Vrijdag']

# ── Employee-data voor animatie ───────────────────────────
_emp_list = []
for _eid, _emp in employees_df.iterrows():
    _emp_list.append({
        'id': int(_eid), 'name': _emp['name'],
        'lat': float(_emp['lat']), 'lon': float(_emp['lon']),
        'dogs': int(_emp['dogs']), 'cats': int(_emp['cats']),
        'smokes': bool(_emp['smokes']),
        'color': EMPLOYEE_COLORS[_eid % len(EMPLOYEE_COLORS)]
    })

# ── Client-data voor animatie ─────────────────────────────
_client_list = []
for _cid, _cl in clients_df.iterrows():
    _unavail = list(_cl['unavailable_day_indices'])
    _client_list.append({
        'id': int(_cid), 'name': _cl['name'],
        'lat': float(_cl['lat']), 'lon': float(_cl['lon']),
        'dogs': int(_cl['dogs']), 'cats': int(_cl['cats']),
        'smokes': bool(_cl['smokes']),
        'tw_start': _cl['time_window_start'], 'tw_end': _cl['time_window_end'],
        'care_hours': float(_cl['care_hours']),
        'unavailable_days': [DAY_NAMES_NL[d] for d in sorted(_unavail)],
        'available_days':   [DAY_NAMES_NL[d] for d in range(5) if d not in _unavail],
    })

# ── Simuleer het zoekproces (zonder emoji’s) ──────────────
_capacity = {_eid: {d: 0 for d in range(5)} for _eid in range(N_EMPLOYEES)}
_events   = []

for _cid, _cl in clients_df.iterrows():
    _c       = _client_list[_cid]
    _unavail = _cl['unavailable_day_indices']
    _assigned = False

    for _emp in _emp_list:
        _eid = _emp['id']
        _base = dict(emp_name=_emp['name'], emp_color=_emp['color'],
                     elat=_emp['lat'], elon=_emp['lon'],
                     client_name=_c['name'], clat=_c['lat'], clon=_c['lon'])

        # Compatibiliteit (tekst ipv emoji)
        _rej = []
        if _emp['dogs'] != -1 and _c['dogs'] > _emp['dogs']:
            _rej.append(('Hond','Compatibiliteit',
                f"Hond: cliënt {_c['dogs']} hond(en), {_emp['name']} max {_emp['dogs']}",
                'compatibility'))
        if _emp['cats'] != -1 and _c['cats'] > _emp['cats']:
            _rej.append(('Kat','Compatibiliteit',
                f"Kat: cliënt {_c['cats']} kat(ten), {_emp['name']} max {_emp['cats']}",
                'compatibility'))
        if not _emp['smokes'] and _c['smokes']:
            _rej.append(('Rook','Compatibiliteit',
                f"{_emp['name']} rookt niet, cliënt rookt wel", 'compatibility'))

        if _rej:
            _ic, _cat, _msg, _chk = _rej[0]
            _events.append({**_base, 'type':'reject','icon':_ic,'reason':_cat,
                            'msg':_msg,'check':_chk,'day':None,
                            'tw':f"{_c['tw_start']}–{_c['tw_end']}",
                            'care':_c['care_hours']})
            continue

        # Per dag
        for _day in range(5):
            _db = dict(**_base, day=DAY_NAMES_NL[_day],
                       tw=f"{_c['tw_start']}–{_c['tw_end']}", care=_c['care_hours'])
            if _day in _unavail:
                _events.append({**_db,'type':'reject','icon':'Beschikb.','reason':'Beschikbaarheid',
                                'msg':f"{_c['name']} niet beschikbaar op {DAY_NAMES_NL[_day]}",
                                'check':'availability'})
                continue
            if _capacity[_eid][_day] >= MAX_CLIENTS_PER_VEHICLE:
                _events.append({**_db,'type':'reject','icon':'Capac.','reason':'Capaciteit',
                                'msg':f"{_emp['name']} heeft al {MAX_CLIENTS_PER_VEHICLE} cliënten op {DAY_NAMES_NL[_day]}",
                                'check':'capacity'})
                continue
            _tw_h = int(_c['tw_end'].split(':')[0]) - int(_c['tw_start'].split(':')[0])
            if _c['care_hours'] > _tw_h:
                _events.append({**_db,'type':'reject','icon':'Venster','reason':'Tijdvenster',
                                'msg':f"Zorgtijd {_c['care_hours']}u past niet in venster ({_tw_h}u)",
                                'check':'timewindow'})
                continue
            # Toegewezen
            _events.append({**_db,'type':'assign','icon':'✓','reason':'Toegewezen',
                            'msg':f"Alle checks geslaagd · {DAY_NAMES_NL[_day]} · {_c['tw_start']}–{_c['tw_end']} · {_c['care_hours']}u",
                            'check':'ok'})
            _capacity[_eid][_day] += 1
            _assigned = True
            break
        if _assigned:
            break

print(f'Animatie-events: {len(_events)} '
      f'(✓ {sum(1 for e in _events if e["type"]=="assign")} toegewezen, '
      f'✗ {sum(1 for e in _events if e["type"]=="reject")} afgewezen)')

# ── HTML sjabloon (zonder emoji’s) ────────────────────────
_center_lat = float(_np.mean([c['lat'] for c in _client_list]))
_center_lon = float(_np.mean([c['lon'] for c in _client_list]))

_ev_js  = _json.dumps(_events)
_emp_js = _json.dumps(_emp_list)
_cl_js  = _json.dumps(_client_list)

_anim_html = f'''<!DOCTYPE html>
<html lang="nl">
<head>
<meta charset="UTF-8">
<title>OR-Tools Zoekproces — Visualisatie</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Mono:ital,wght@0,400;0,500;1,400&family=Syne:wght@700;800;900&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
:root {{
  --bg:       #080b10;
  --surface:  #0e1420;
  --surface2: #151e2e;
  --border:   #1f2d42;
  --text:     #dbe4f0;
  --dim:      #4a5f7a;
  --accent:   #3b82f6;
  --green:    #22c55e;
  --red:      #f43f5e;
  --yellow:   #f59e0b;
  --purple:   #a78bfa;
  --cyan:     #22d3ee;
  --mono:     'DM Mono', monospace;
  --display:  'Syne', sans-serif;
}}
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
html, body {{ height: 100%; background: var(--bg); color: var(--text); font-family: var(--mono); overflow: hidden; }}

/* ── Layout ───────────────────────────────────────── */
#app {{
  display: grid;
  grid-template-rows: 56px 1fr;
  grid-template-columns: 1fr 380px;
  height: 100vh;
}}

/* ── Header ── */
#hdr {{
  grid-column: 1/-1;
  background: var(--surface);
  border-bottom: 1px solid var(--border);
  display: flex; align-items: center; gap: 16px;
  padding: 0 20px;
}}
#hdr h1 {{
  font-family: var(--display); font-size: 14px; font-weight: 900;
  letter-spacing: .06em; text-transform: uppercase;
  background: linear-gradient(90deg, #60a5fa, #a78bfa);
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  white-space: nowrap;
}}
.badge {{
  background: var(--surface2); border: 1px solid var(--border);
  border-radius: 3px; padding: 2px 8px; font-size: 10px; color: var(--dim);
  letter-spacing: .04em;
}}
.sep {{ flex: 1; }}
.ctrl {{ display: flex; align-items: center; gap: 8px; }}
.ctrl-lbl {{ font-size: 10px; color: var(--dim); letter-spacing: .05em; text-transform: uppercase; }}
input[type=range] {{
  -webkit-appearance: none; width: 80px; height: 2px;
  border-radius: 2px; background: var(--border); outline: none; cursor: pointer;
}}
input[type=range]::-webkit-slider-thumb {{
  -webkit-appearance: none; width: 11px; height: 11px;
  border-radius: 50%; background: var(--accent); cursor: pointer;
}}
#speed-val {{ font-size: 11px; color: var(--accent); min-width: 24px; }}
.btn {{
  background: var(--surface2); border: 1px solid var(--border);
  color: var(--text); font-family: var(--mono); font-size: 11px;
  padding: 6px 14px; border-radius: 4px; cursor: pointer;
  transition: background .12s, border-color .12s; white-space: nowrap;
}}
.btn:hover {{ background: var(--border); }}
.btn.primary {{ background: var(--accent); border-color: var(--accent); color: #fff; }}
.btn.primary:hover {{ background: #2563eb; }}
.btn.primary.paused {{ background: var(--yellow); border-color: var(--yellow); color: #000; }}

/* ── Map ── */
#map-wrap {{ position: relative; overflow: hidden; }}
#map-el {{ width: 100%; height: 100%; }}
.leaflet-container {{ background: var(--bg) !important; }}
.leaflet-tile {{ filter: brightness(.28) saturate(.3) hue-rotate(200deg); }}

/* Progress */
#prog {{ position: absolute; bottom: 0; left: 0; right: 0; height: 3px; background: var(--border); z-index: 900; }}
#prog-fill {{ height: 100%; width: 0; background: linear-gradient(90deg,#3b82f6,#22d3ee,#22c55e); transition: width .08s; }}

/* Counter overlay on map */
#map-counter {{
  position: absolute; top: 12px; left: 12px; z-index: 900;
  background: rgba(8,11,16,.82); border: 1px solid var(--border);
  border-radius: 6px; padding: 10px 14px; backdrop-filter: blur(6px);
}}
.mc-row {{ display: flex; align-items: center; gap: 8px; font-size: 12px; margin-bottom: 4px; }}
.mc-row:last-child {{ margin-bottom: 0; }}
.mc-dot {{ width: 7px; height: 7px; border-radius: 50%; flex-shrink: 0; }}
.mc-num {{ font-family: var(--display); font-size: 18px; font-weight: 800; min-width: 40px; }}

/* ── Right panel ── */
#panel {{
  background: var(--surface); border-left: 1px solid var(--border);
  display: flex; flex-direction: column; overflow: hidden;
}}

/* Legend / filter strip */
#legend {{
  padding: 10px 14px; border-bottom: 1px solid var(--border);
  display: flex; gap: 8px; flex-wrap: wrap;
}}
.leg-item {{
  display: flex; align-items: center; gap: 5px;
  font-size: 10px; color: var(--dim); cursor: pointer;
  padding: 3px 7px; border-radius: 3px; border: 1px solid transparent;
  transition: all .12s; user-select: none;
}}
.leg-item:hover {{ border-color: var(--border); color: var(--text); }}
.leg-item.active {{ border-color: var(--border); color: var(--text); background: var(--surface2); }}
.leg-dot {{ width: 8px; height: 8px; border-radius: 2px; flex-shrink: 0; }}

/* Current event */
#cur {{
  padding: 14px; border-bottom: 1px solid var(--border);
  min-height: 118px; flex-shrink: 0;
}}
.sec-lbl {{
  font-size: 9px; color: var(--dim); letter-spacing: .1em;
  text-transform: uppercase; margin-bottom: 8px;
}}
#ev-card {{
  background: var(--surface2); border-radius: 5px;
  padding: 11px 13px; border-left: 3px solid var(--border);
  transition: border-color .18s;
}}
#ev-card.assign {{ border-left-color: var(--green); }}
#ev-card.reject-compat {{ border-left-color: var(--red); }}
#ev-card.reject-avail  {{ border-left-color: var(--yellow); }}
#ev-card.reject-cap    {{ border-left-color: var(--purple); }}
#ev-card.reject-tw     {{ border-left-color: var(--cyan); }}

.ev-top {{ display: flex; align-items: center; gap: 8px; margin-bottom: 5px; }}
.ev-icon-big {{ font-size: 20px; font-weight: bold; min-width: 32px; }}
.ev-reason {{ font-size: 10px; letter-spacing: .07em; text-transform: uppercase; font-weight: 500; }}
.ev-msg {{ font-size: 11.5px; line-height: 1.5; color: var(--text); }}
.ev-meta {{ margin-top: 7px; font-size: 10.5px; color: var(--dim); display: flex; gap: 10px; flex-wrap: wrap; }}
.ev-meta .hl {{ color: var(--text); }}
.ev-meta .emp-name {{ font-weight: 500; }}

/* Log */
#log-hdr {{
  padding: 9px 14px; border-bottom: 1px solid var(--border);
  display: flex; align-items: center; justify-content: space-between;
  font-size: 9px; color: var(--dim); letter-spacing: .1em; text-transform: uppercase;
  flex-shrink: 0;
}}
#log {{ flex: 1; overflow-y: auto; padding: 4px 6px; }}
#log::-webkit-scrollbar {{ width: 3px; }}
#log::-webkit-scrollbar-thumb {{ background: var(--border); border-radius: 2px; }}

.lr {{
  display: flex; gap: 8px; align-items: flex-start;
  padding: 5px 8px; border-radius: 4px; font-size: 11px; line-height: 1.4;
  animation: slideIn .12s ease;
  cursor: default;
}}
@keyframes slideIn {{ from {{ opacity:0; transform:translateY(3px); }} to {{ opacity:1; }} }}
.lr:hover {{ background: var(--surface2); }}
.lr-dot {{ width: 6px; height: 6px; border-radius: 50%; flex-shrink:0; margin-top:4px; }}
.lr-body {{ flex: 1; }}
.lr-names {{ color: var(--text); }}
.lr-names .ename {{ font-weight: 500; }}
.lr-detail {{ color: var(--dim); font-size: 10.5px; }}
.lr-ico {{ font-size: 12px; flex-shrink:0; margin-top:1px; font-weight: bold; }}
</style>
</head>
<body>
<div id="app">

  <!-- Header -->
  <div id="hdr">
    <h1>OR‑Tools Zoekproces</h1>
    <span class="badge">20 medewerkers</span>
    <span class="badge">100 cliënten</span>
    <span class="badge">5 dagen</span>
    <div class="sep"></div>
    <div class="ctrl">
      <span class="ctrl-lbl">Snelheid</span>
      <input type="range" id="speed" min="1" max="12" value="6">
      <span id="speed-val">6×</span>
    </div>
    <button class="btn primary" id="play-btn">▶ Start</button>
    <button class="btn" id="reset-btn">↺ Reset</button>
  </div>

  <!-- Map -->
  <div id="map-wrap">
    <div id="map-el"></div>
    <div id="map-counter">
      <div class="mc-row">
        <div class="mc-dot" style="background:#22c55e"></div>
        <div class="mc-num" id="cnt-assign" style="color:#22c55e">0</div>
        <span style="font-size:11px;color:#4a5f7a">Toegewezen</span>
      </div>
      <div class="mc-row">
        <div class="mc-dot" style="background:#f43f5e"></div>
        <div class="mc-num" id="cnt-reject" style="color:#f43f5e">0</div>
        <span style="font-size:11px;color:#4a5f7a">Afgewezen</span>
      </div>
      <div class="mc-row">
        <div class="mc-dot" style="background:#3b82f6"></div>
        <div class="mc-num" id="cnt-step"   style="color:#3b82f6">0</div>
        <span style="font-size:11px;color:#4a5f7a">van {len(_events)}</span>
      </div>
    </div>
    <div id="prog"><div id="prog-fill"></div></div>
  </div>

  <!-- Side panel -->
  <div id="panel">

    <div id="legend">
      <div class="leg-item active" data-check="all">Alles</div>
      <div class="leg-item" data-check="compatibility" style="--c:#f43f5e">
        <div class="leg-dot" style="background:#f43f5e"></div>Huisdier/Rook
      </div>
      <div class="leg-item" data-check="availability" style="--c:#f59e0b">
        <div class="leg-dot" style="background:#f59e0b"></div>Beschikbaarheid
      </div>
      <div class="leg-item" data-check="capacity" style="--c:#a78bfa">
        <div class="leg-dot" style="background:#a78bfa"></div>Capaciteit
      </div>
      <div class="leg-item" data-check="ok" style="--c:#22c55e">
        <div class="leg-dot" style="background:#22c55e"></div>Toegewezen
      </div>
    </div>

    <div id="cur">
      <div class="sec-lbl">Huidige poging</div>
      <div id="ev-card">
        <div style="color:var(--dim);font-size:12px;padding:4px 0;">Druk op ▶ Start om te beginnen…</div>
      </div>
    </div>

    <div id="log-hdr">
      <span>Activiteitenlog</span>
      <span id="log-cnt">0 events</span>
    </div>
    <div id="log"></div>

  </div>
</div>

<script>
const EVENTS    = {_ev_js};
const EMPLOYEES = {_emp_js};
const CLIENTS   = {_cl_js};
const CENTER    = [{_center_lat}, {_center_lon}];
const TOTAL     = EVENTS.length;

/* ── Colour helpers ─────────────────────────────── */
const CHECK_COLOR = {{
  compatibility: '#f43f5e',
  availability:  '#f59e0b',
  capacity:      '#a78bfa',
  timewindow:    '#22d3ee',
  ok:            '#22c55e',
}};
const CHECK_CLASS = {{
  compatibility: 'reject-compat',
  availability:  'reject-avail',
  capacity:      'reject-cap',
  timewindow:    'reject-tw',
  ok:            'assign',
}};

/* ── Map ────────────────────────────────────────── */
const map = L.map('map-el', {{ center: CENTER, zoom: 14, zoomControl: true, attributionControl: false }});
L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_matter/{{z}}/{{x}}/{{y}}{{r}}.png',
  {{ subdomains: 'abcd', maxZoom: 19 }}).addTo(map);

/* Static employee markers (tooltip zonder emoji) */
EMPLOYEES.forEach(emp => {{
  const el = document.createElement('div');
  el.style.cssText = `width:18px;height:18px;border-radius:50%;background:${{emp.color}};
    border:3px solid rgba(255,255,255,.75);box-shadow:0 2px 8px rgba(0,0,0,.7);`;
  L.marker([emp.lat, emp.lon],
    {{ icon: L.divIcon({{ html: el, iconSize:[18,18], iconAnchor:[9,9], className:'' }}) }})
   .bindTooltip(`<b style="font-size:12px">${{emp.name}}</b><br>
     Hond: ${{emp.dogs===-1?'geen voorkeur':'max '+emp.dogs}} &nbsp;
     Kat: ${{emp.cats===-1?'geen voorkeur':'max '+emp.cats}} &nbsp;
     Roken: ${{emp.smokes?'rookt':'rookt niet'}}`,
     {{ direction:'top', className:'leaflet-tooltip' }})
   .addTo(map);
}});

/* Static client dots (very dim) */
const clientMarkers = {{}};
CLIENTS.forEach(cl => {{
  const m = L.circleMarker([cl.lat, cl.lon],
    {{ radius:4, color:'#1e3a52', weight:1.5, fillColor:'#1e3a52', fillOpacity:.8 }}).addTo(map);
  clientMarkers[cl.name] = m;
}});

/* ── State ──────────────────────────────────────── */
let step = 0, playing = false, timer = null;
let assigns = 0, rejects = 0;
let activeLayer = null, pulseLayer = null;
let filterCheck = 'all';
const LOG_MAX = 100;

/* ── Speed → ms delay ───────────────────────────── */
function delay() {{
  const v = +document.getElementById('speed').value;
  return Math.round(700 / (v * 0.85 + 0.15));
}}

/* ── Draw animated arrow ────────────────────────── */
function drawArrow(elat, elon, clat, clon, color, isDash) {{
  if (activeLayer) {{ activeLayer.remove(); activeLayer = null; }}
  const line = L.polyline([[elat,elon],[clat,clon]], {{
    color, weight: isDash ? 1.5 : 2.5, opacity: .88,
    dashArray: isDash ? '5 6' : null,
  }}).addTo(map);
  const glow = L.circleMarker([clat,clon], {{
    radius: 10, color, weight: 2, fillColor: color, fillOpacity: .18
  }}).addTo(map);
  activeLayer = {{ remove() {{ line.remove(); glow.remove(); }} }};
}}

/* ── Pulse assigned client ──────────────────────── */
function pulseClient(name, color) {{
  if (pulseLayer) {{ pulseLayer.remove(); pulseLayer = null; }}
  const cl = CLIENTS.find(c => c.name === name);
  if (!cl) return;
  if (clientMarkers[name]) {{
    clientMarkers[name].setStyle({{ color, fillColor: color, fillOpacity: .9 }});
  }}
  pulseLayer = L.circleMarker([cl.lat, cl.lon], {{
    radius: 13, color, weight: 2, fillColor: color, fillOpacity: .15
  }}).addTo(map);
}}

/* ── Render one event ───────────────────────────── */
function renderEvent(ev) {{
  const isAssign = ev.type === 'assign';
  const cc = CHECK_COLOR[ev.check] || (isAssign ? '#22c55e' : '#f43f5e');
  const isDash = !isAssign;

  drawArrow(ev.elat, ev.elon, ev.clat, ev.clon, cc, isDash);
  if (isAssign) pulseClient(ev.client_name, cc);

  // Stats
  document.getElementById('cnt-step').textContent   = step + 1;
  document.getElementById('cnt-assign').textContent = assigns;
  document.getElementById('cnt-reject').textContent = rejects;
  document.getElementById('prog-fill').style.width  = ((step+1)/TOTAL*100).toFixed(2) + '%';

  // Current card
  const card = document.getElementById('ev-card');
  card.className = 'ev-' + (isAssign ? 'card assign' : 'card ' + (CHECK_CLASS[ev.check] || 'reject-compat'));
  card.innerHTML = `
    <div class="ev-top">
      <span class="ev-icon-big">${{ev.icon}}</span>
      <span class="ev-reason" style="color:${{cc}}">${{ev.reason}}</span>
    </div>
    <div class="ev-msg">${{ev.msg}}</div>
    <div class="ev-meta">
      <span><span class="hl emp-name" style="color:${{ev.emp_color}}">${{ev.emp_name}}</span></span>
      <span>→ <span class="hl">${{ev.client_name}}</span></span>
      ${{ev.day ? `<span style="color:var(--dim)">📅 ${{ev.day}}</span>` : ''}}
      <span style="color:var(--dim)">⏱ ${{ev.tw}}</span>
      <span style="color:var(--dim)">🏠 ${{ev.care}}u</span>
    </div>`;

  // Log (skip if filtered out)
  if (filterCheck === 'all' || ev.check === filterCheck) {{
    const row = document.createElement('div');
    row.className = 'lr';
    row.innerHTML = `
      <div class="lr-dot" style="background:${{ev.emp_color}}"></div>
      <div class="lr-body">
        <div class="lr-names">
          <span class="ename" style="color:${{ev.emp_color}}">${{ev.emp_name}}</span>
          <span style="color:var(--dim)"> → </span>
          <span>${{ev.client_name}}</span>
          <span class="lr-ico" style="margin-left:6px">${{ev.icon}}</span>
        </div>
        <div class="lr-detail">${{ev.msg}}${{ev.day ? ' · '+ev.day : ''}}</div>
      </div>`;
    const log = document.getElementById('log');
    log.insertBefore(row, log.firstChild);
    while (log.children.length > LOG_MAX) log.removeChild(log.lastChild);
    document.getElementById('log-cnt').textContent = Math.min(step+1, LOG_MAX) + ' events';
  }}
}}

/* ── Tick ───────────────────────────────────────── */
function tick() {{
  if (step >= TOTAL) {{
    playing = false;
    document.getElementById('play-btn').textContent = '✓ Voltooid';
    document.getElementById('play-btn').className   = 'btn';
    if (activeLayer) activeLayer.remove();
    return;
  }}
  const ev = EVENTS[step];
  if (ev.type === 'assign') assigns++; else rejects++;
  renderEvent(ev);
  step++;
  if (playing) timer = setTimeout(tick, delay());
}}

/* ── Controls ───────────────────────────────────── */
document.getElementById('play-btn').addEventListener('click', () => {{
  if (step >= TOTAL) return;
  playing = !playing;
  const btn = document.getElementById('play-btn');
  if (playing) {{
    btn.textContent = '⏸ Pauze';
    btn.className   = 'btn primary paused';
    tick();
  }} else {{
    btn.textContent = '▶ Doorgaan';
    btn.className   = 'btn primary';
    clearTimeout(timer);
  }}
}});

document.getElementById('reset-btn').addEventListener('click', () => {{
  playing = false; clearTimeout(timer);
  step = 0; assigns = 0; rejects = 0;
  if (activeLayer) {{ activeLayer.remove(); activeLayer = null; }}
  if (pulseLayer)  {{ pulseLayer.remove();  pulseLayer  = null; }}
  CLIENTS.forEach(cl => {{
    if (clientMarkers[cl.name])
      clientMarkers[cl.name].setStyle({{ color:'#1e3a52', fillColor:'#1e3a52', fillOpacity:.8 }});
  }});
  document.getElementById('play-btn').textContent = '▶ Start';
  document.getElementById('play-btn').className   = 'btn primary';
  ['cnt-step','cnt-assign','cnt-reject'].forEach(id => document.getElementById(id).textContent = '0');
  document.getElementById('prog-fill').style.width = '0%';
  document.getElementById('ev-card').className = 'ev-card';
  document.getElementById('ev-card').innerHTML = '<div style="color:var(--dim);font-size:12px;padding:4px 0;">Druk op ▶ Start om te beginnen…</div>';
  document.getElementById('log').innerHTML = '';
  document.getElementById('log-cnt').textContent = '0 events';
}});

document.getElementById('speed').addEventListener('input', function() {{
  document.getElementById('speed-val').textContent = this.value + '×';
}});

/* ── Filter tabs ─────────────────────────────────── */
document.querySelectorAll('.leg-item').forEach(el => {{
  el.addEventListener('click', () => {{
    document.querySelectorAll('.leg-item').forEach(x => x.classList.remove('active'));
    el.classList.add('active');
    filterCheck = el.dataset.check;
  }});
}});
</script>
</body>
</html>'''

# Schrijf het bestand weg
with open('../output/vrp_process_animation.html', 'w', encoding='utf-8') as f:
    f.write(_anim_html)

print('\nAnimatie gegenereerd (zonder emoji’s): ../output/vrp_process_animation.html')
display(HTML('<a href="../output/vrp_process_animation.html" target="_blank">▶ Open OR-Tools animatie</a>'))

Animatie-events: 1442 (✓ 67 toegewezen, ✗ 1375 afgewezen)

Animatie gegenereerd (zonder emoji’s): ../output/vrp_process_animation.html
